# Pancreatic cancer survival: does methylation add anything over clinical data?

Redo of the 2023 practicum analysis (`docs/2023-practicum/`) on a clean cohort:

* **TCGA-PAAD primary tumors only, 450K array, one sample per patient.** The 2023 run pooled tumor, adjacent-normal and organoid samples from three projects on two array platforms, with some patients counted more than once.
* **A clinical baseline** (age + stage) to compare against.
* **All preprocessing (probe selection, scaling, PCA, penalty tuning) fit inside each CV fold**, scored with Harrell's C, Uno's C, time-dependent AUC and integrated Brier score.

Run the data steps once from the repo root:

```bash
methylspan files --site pancreas --platform 450K --out data/paad_450k_files.csv
methylspan download data/paad_450k_files.csv            # ~2.4 GB, or reuse an old download:
# methylspan matrix ... --raw-dir path/to/old_download --resolve-superseded
methylspan matrix data/paad_450k_files.csv --out data/paad_450k_betas.parquet
methylspan clinical data/paad_450k_files.csv --out data/paad_clinical.csv
```

In [ ]:
import pandas as pd
from methylspan import matrix, survival

DATA = "../data"
clinical = pd.read_csv(f"{DATA}/paad_clinical.csv", index_col="case_id")
betas = matrix.filter_probes(matrix.load_matrix(f"{DATA}/paad_450k_betas.parquet"), max_missing=0.1)
X, y = survival.assemble(clinical, betas.T)
print(f"{len(X)} patients, {y['event'].sum()} deaths, {betas.shape[0]} probes")

In [ ]:
clinical["primary_diagnosis"].value_counts()

TCGA-PAAD includes some non-ductal tumors (e.g. neuroendocrine). To restrict to ductal adenocarcinoma, filter `clinical` on `primary_diagnosis` before `assemble`.

In [ ]:
models = {
    "clinical (age+stage)": survival.make_model(use_methylation=False),
    "methylation PCA-15 (2023-style)": survival.make_model(use_clinical=False, n_top=betas.shape[0], n_components=15),
    "clinical + PCA-15": survival.make_model(n_top=betas.shape[0], n_components=15),
    "clinical + coxnet top-5k": survival.tune(survival.make_model(n_top=5000, estimator=survival.coxnet())),
}
results = survival.compare_models(models, X, y, n_splits=5, n_repeats=3)
survival.summarize(results)

Higher C-index / AUC is better (0.5 = random); lower integrated Brier score is better.

## Next steps
* Validate on CPTAC-3 primary tumors (EPIC; restrict to probes shared with 450K).
* Replace PCs with interpretable features: tumor purity, immune/stromal fractions from reference-based deconvolution, epigenetic age, promoter methylation of a-priori genes (`methylspan.mapping.aggregate_by_gene`).